# Testar RAG — playground de perguntas

Pra testar uma pergunta qualquer em cima dos documentos da pesquisa e ver **se o RAG acha a resposta que você quer**.

Usa o mesmo pipeline dos experimentos (`scripts/pipeline_base.py`): mesmos chunks (1000/100), mesmo embedding (BGE-M3), mesmos prompts dos 6 métodos.

**Como usar**
1. Rode a célula **0. Setup** uma vez.
2. Na célula **1. Documento**, escolha o domínio e rode.
3. Na célula **2. Pergunta**, troque `QUERY` e `TERMOS` (o que você espera encontrar), e rode as células **2, 3 e 4**.

- A célula **3. Recuperação** é local e gratuita: mostra os chunks recuperados e diz se o termo que você procura veio no top-k, se existe no documento mas ficou de fora (falha de recuperação), ou se nem existe no documento.
- A célula **4. Geração** chama o LLM (pago, se o backend for OpenAI).
- **Nada aqui é gravado nos JSONL dos experimentos**, e o índice do notebook fica numa pasta separada (`notebooks/.chroma_notebook/`, fora do git).

In [15]:
# ===== 0. Setup (rodar uma vez) =====
import os
import re
import sys
import time
import unicodedata
from pathlib import Path

import pandas as pd

REPO_DIR = Path.cwd().resolve()
while not (REPO_DIR / "Pesquisa" / "scripts").exists() and REPO_DIR != REPO_DIR.parent:
    REPO_DIR = REPO_DIR.parent
sys.path.insert(0, str(REPO_DIR / "Pesquisa" / "scripts"))

# Mesmo embedding dos experimentos -- precisa ser definido ANTES de importar o pipeline_base
os.environ.setdefault("EMBEDDING_MODEL_NAME", "BAAI/bge-m3")
from dotenv import load_dotenv
load_dotenv(REPO_DIR / ".env")

import pipeline_base as pb
from comparar_metodos import carregar_grendene, carregar_revalida, carregar_sbc

# O índice do notebook fica separado do dos experimentos (Pesquisa/vectorstores, que é versionado)
pb.VECTORSTORE_DIR = REPO_DIR / "Pesquisa" / "notebooks" / ".chroma_notebook"

METODOS_COM_CONTEXTO = {
    "stuff": pb.gerar_resposta_stuff,
    "refine": pb.gerar_resposta_refine,
    "map_reduce": pb.gerar_resposta_map_reduce,
    "map_rerank": pb.gerar_resposta_map_rerank,
}
METODOS_NIVEL_QUERY = {
    "query_step_down": pb.gerar_resposta_query_step_down,
    "reciprocal": pb.gerar_resposta_reciprocal,
}

_VARIANTES = {"a": "aáàâã", "e": "eéèê", "i": "iíì", "o": "oóòôõ", "u": "uúùü", "c": "cç"}
_DESTAQUE, _FIM = "\033[1;30;43m", "\033[0m"


def _regex_termo(termo: str) -> re.Pattern:
    """Busca tolerante: ignora maiúsculas, acentos e quebras de linha no meio do trecho
    (o PDF->txt costuma quebrar frases no meio)."""
    base = unicodedata.normalize("NFKD", re.sub(r"\s+", " ", termo.strip().lower()))
    base = "".join(ch for ch in base if not unicodedata.combining(ch))
    partes = []
    for ch in base:
        if ch == " ":
            partes.append(r"\s+")
        elif ch in _VARIANTES:
            partes.append(f"[{_VARIANTES[ch]}]")
        else:
            partes.append(re.escape(ch))
    return re.compile("".join(partes), re.IGNORECASE)


def contem(texto: str, termo: str) -> bool:
    return bool(_regex_termo(termo).search(texto))


def destacar(texto: str, termos: list[str]) -> str:
    for termo in termos:
        texto = _regex_termo(termo).sub(lambda m: f"{_DESTAQUE}{m.group(0)}{_FIM}", texto)
    return texto


def obter_colecao(chunks: list[dict], nome: str):
    """Reaproveita o índice se ele já tem exatamente esses chunks -- evita reencodar o documento
    toda vez que o kernel reinicia."""
    import chromadb

    client = chromadb.PersistentClient(path=str(pb.VECTORSTORE_DIR))
    try:
        colecao = client.get_collection(nome)
        pontas = [chunks[0], chunks[-1]]
        guardados = colecao.get(ids=[f"{c['document_id']}#{c['chunk_index']}" for c in pontas])["documents"]
        if colecao.count() == len(chunks) and sorted(guardados) == sorted(c["text"] for c in pontas):
            return colecao
    except Exception:
        pass
    print(f"Indexando {len(chunks)} chunks (só na primeira vez)...")
    return pb.construir_indice(chunks, colecao_nome=nome)


modelo_atual = pb.OPENAI_MODEL if pb.LLM_BACKEND == "openai" else pb.OLLAMA_MODEL
print(f"Embedding: {pb.EMBEDDING_MODEL_NAME} | LLM: {pb.LLM_BACKEND} / {modelo_atual}")

Embedding: BAAI/bge-m3 | LLM: openai / gpt-4o-mini


## 1. Documento

In [16]:
# ======= ESCOLHA O DOCUMENTO =======
DOMINIO = "revalida"  # "revalida" | "grendene" | "sbc" | "arquivo"
ARQUIVO_TXT = REPO_DIR / "Pesquisa/dados/brutos/sbp_consulta_adolescente.txt"  # só usado se DOMINIO = "arquivo"
# ===================================

CARREGADORES = {"revalida": carregar_revalida, "grendene": carregar_grendene, "sbc": carregar_sbc}

if DOMINIO == "arquivo":
    stem = Path(ARQUIVO_TXT).stem
    chunks = pb.carregar_e_chunkar(str(ARQUIVO_TXT), document_id=stem)
    perguntas_dataset = []
    nome_colecao = "arquivo_" + re.sub(r"[^a-zA-Z0-9_-]", "_", stem)[:50]
else:
    _, chunks, perguntas_dataset = CARREGADORES[DOMINIO]()
    nome_colecao = DOMINIO

colecao = obter_colecao(chunks, nome_colecao)
chunk_por_texto = {c["text"]: c for c in chunks}
print(f"{DOMINIO}: {len(chunks)} chunks indexados.")

if perguntas_dataset:
    print(f"\nPerguntas do dataset (use o id em PERGUNTA_DO_DATASET, na célula 2):")
    for p in perguntas_dataset:
        print(f"  {p['id']}: {p['query'].splitlines()[-1][:110]}")

Indexando 430 chunks (só na primeira vez)...


Inference Embeddings: 100%|██████████| 2/2 [08:06<00:00, 243.04s/it]


revalida: 430 chunks indexados.

Perguntas do dataset (use o id em PERGUNTA_DO_DATASET, na célula 2):
  revalida_q1_a: Qual é o diagnóstico atual mais provável para explicar o agravamento do quadro respiratório?
  revalida_q1_b: Cite três sinais clínicos que, se presentes no seguimento do exame físico da paciente, corroborariam o diagnós
  revalida_q3_a: Explique se a paciente poderá ser atendida sem a presença de um responsável maior de idade, considerando suas 
  revalida_q3_b: Caso seja confirmada a gravidez da paciente, qual deverá ser a conduta do médico quanto ao atendimento da adol
  revalida_q3_c: Considere que a unidade onde ocorreu a consulta promova grupos de discussão em saúde e que já estão agendadas 
  revalida_q4_a: Fundamentado nas recomendações da Organização Mundial da Saúde (2018), qual é o diagnóstico correto do caso cl
  revalida_q4_b: Fundamentado nas recomendações da Organização Mundial da Saúde (2018), é possível constatar sinais de progress
  revalida_q4_c: Qua

## 2. Pergunta — troque aqui

In [21]:
# ======= TROQUE AQUI =======
QUERY = "Qual a idade máxima para registrar um procedimento na pediatria?"
TERMOS = ["5,3186"]       # o que você espera achar no documento (número, palavra, trecho). [] = não checar
RESPOSTA_ESPERADA = None  # opcional: gabarito, pra calcular a similaridade de cosseno da resposta
K = 4                     # nº de chunks recuperados (os experimentos usam 4)

PERGUNTA_DO_DATASET = None  # ex.: "grendene_q3" -> usa a QUERY e a RESPOSTA_ESPERADA do dataset
# ===========================

if PERGUNTA_DO_DATASET:
    p = next(p for p in perguntas_dataset if p["id"] == PERGUNTA_DO_DATASET)
    QUERY, RESPOSTA_ESPERADA = p["query"], p["esperada"]

print(QUERY)
if RESPOSTA_ESPERADA:
    print(f"\nEsperada: {RESPOSTA_ESPERADA}")

Qual a idade máxima para registrar um procedimento na pediatria?


## 3. Recuperação (local, grátis)

Mostra os `K` chunks que o RAG entregaria pro LLM e, pra cada termo de `TERMOS`, diz onde ele está:
- ✅ veio no top-K: a recuperação funcionou, o resto é com o LLM;
- ⚠️ existe no documento, mas ficou de fora do top-K: **falha de recuperação** (o LLM nunca vai ver);
- ❌ não existe no documento: confira a grafia/formatação (ex.: `5.3186` vs `5,3186`) ou se o PDF→txt perdeu o trecho.

In [22]:
RANKING_MAX = 50  # até onde procurar a posição de um termo que não veio no top-K

top_k = pb.recuperar_com_similaridade(colecao, QUERY, k=K)
ranking = [texto for texto, _ in pb.recuperar_com_similaridade(colecao, QUERY, k=min(RANKING_MAX, colecao.count()))]

print(f"Top-{K} para: {QUERY.splitlines()[-1]}\n")
for pos, (texto, sim) in enumerate(top_k, start=1):
    c = chunk_por_texto.get(texto, {})
    achados = [t for t in TERMOS if contem(texto, t)]
    marca = f"  ✅ contém: {', '.join(achados)}" if achados else ""
    print(f"━━━ #{pos}  sim={sim:.3f}  [{c.get('document_id')}#{c.get('chunk_index')}]{marca}")
    print(destacar(texto, TERMOS))
    print()

if TERMOS:
    print("━━━ Diagnóstico dos termos")
    textos_top_k = [texto for texto, _ in top_k]
    for termo in TERMOS:
        no_documento = [c for c in chunks if contem(c["text"], termo)]
        posicoes = [ranking.index(c["text"]) + 1 for c in no_documento if c["text"] in ranking]
        if any(contem(t, termo) for t in textos_top_k):
            print(f'  ✅ "{termo}": veio no top-{K} (posição {min(posicoes) if posicoes else "?"}).')
        elif no_documento:
            onde = f"melhor posição no ranking: {min(posicoes)}" if posicoes else f"fora do top-{RANKING_MAX}"
            ids = ", ".join(f"#{c['chunk_index']}" for c in no_documento[:10])
            print(f'  ⚠️ "{termo}": existe no documento ({len(no_documento)} chunk(s): {ids}), mas não veio no '
                  f"top-{K} ({onde}) -> falha de recuperação.")
        else:
            print(f'  ❌ "{termo}": não aparece em nenhum chunk do documento.')

Top-4 para: Qual a idade máxima para registrar um procedimento na pediatria?

━━━ #1  sim=0.630  [sbp_consulta_adolescente#27]
20 anos incompletos).
Em relação às questões éticas, no Artigo 4º e 
no Parágrafo Único do Artigo 5º do Código Civil 
Brasileiro e quanto ao referencial bioético da Au-
tonomia, as pessoas maiores de 16 anos poderão 
optar pelo atendimento por médico não pediatra.
Pelo viés da ética, os Departamentos de Bio-
ética e de Pediatria Legal da Sociedade de Pe-
diatria de São Paulo, com reconhecimento da Câ-
mara Técnica de Pediatria do Conselho Regional 
de Medicina do Estado de São Paulo – CREMESP 
– também recomendam:
• Os estabelecimentos de Prontos-Socorros Pú-
blicos e Privados deverão ser estruturados para 
prestar atendimento às situações de urgência e 
emergência e estão obrigados a manter, em re-
gime de plantão no local, profissionais da área 
da Pediatria.
• A obrigação de manter o profissional especia-
lista é de responsabilidade do gestor do Pron-
to-Soc

## 4. Geração (chama o LLM)

Chamadas ao LLM por pergunta: Stuff 1 · Refine K · Map-Reduce K+1 · Map-Rerank K · Query Step-Down 6 · Reciprocal 5.

In [23]:
# ======= ESCOLHA OS MÉTODOS =======
METODOS = ["stuff"]  # qualquer combinação de: "stuff", "refine", "map_reduce", "map_rerank", "query_step_down", "reciprocal"
LIMIAR_COMPRESSAO = 0.0  # 0.0 = sem filtro; ex.: 0.7 descarta chunks com sim < 0,7 (não vale pra query_step_down/reciprocal)
MODELO_LLM = "gpt-4o-mini"  # só vale com LLM_BACKEND=openai; troque pra ver se outro modelo responde diferente
# ==================================

pb.OPENAI_MODEL = MODELO_LLM
contextos = pb.aplicar_compressao_contextual(pb.recuperar_com_similaridade(colecao, QUERY, k=K), LIMIAR_COMPRESSAO)
termo_no_contexto = any(contem(t, termo) for t in contextos for termo in TERMOS)

linhas = []
for metodo in METODOS:
    inicio = time.time()
    if metodo in METODOS_NIVEL_QUERY:
        r = METODOS_NIVEL_QUERY[metodo](QUERY, colecao, k=K)
    else:
        r = METODOS_COM_CONTEXTO[metodo](QUERY, contextos)
    tempo = time.time() - inicio
    resposta = r["resposta_texto"]
    achados = [t for t in TERMOS if contem(resposta, t)]
    sim = pb.calcular_similaridade(resposta, RESPOSTA_ESPERADA) if RESPOSTA_ESPERADA else None

    print(f"━━━ {metodo}  ({r['tokens_prompt'] + r['tokens_gerados']} tokens, {tempo:.1f}s)")
    if r.get("perguntas_alternativas"):
        print("Perguntas alternativas geradas:")
        for alt in r["perguntas_alternativas"]:
            print(f"  - {alt}")
    print(destacar(resposta, TERMOS))
    if TERMOS and not achados and termo_no_contexto and metodo in METODOS_COM_CONTEXTO:
        print("⚠️ O termo estava no contexto entregue ao LLM, mas não apareceu na resposta.")
    print()

    linhas.append({
        "método": metodo,
        "termos na resposta": f"{len(achados)}/{len(TERMOS)}" if TERMOS else "-",
        "sim. cosseno": round(sim, 3) if sim is not None else None,
        "tokens": r["tokens_prompt"] + r["tokens_gerados"],
        "tempo (s)": round(tempo, 1),
        "chunks": len(contextos) if metodo in METODOS_COM_CONTEXTO else "próprios",
    })

pd.DataFrame(linhas)

━━━ stuff  (1093 tokens, 0.7s)
A idade máxima para registrar um procedimento na pediatria é de 19 anos e 11 meses e 29 dias.



,método,termos na resposta,sim. cosseno,tokens,tempo (s),chunks
0,stuff,0/1,None,1093,0.7,4
